In [5]:
!pip install dagshub mlflow --quiet

import os
import gc
import json
import pickle
import warnings
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, confusion_matrix

!pip install mlflow dagshub --quiet

import dagshub
import mlflow
import mlflow.sklearn

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

In [6]:
dagshub.init(
    repo_owner="ChorniBero15",
    repo_name="ML2",
    mlflow=True
)

mlflow.set_experiment("DecisionTree_Training")

REGISTERED_MODEL_NAME = "model_decision_tree"

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=fb907cd5-133f-4c1c-bacc-34515aa511f8&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=4d3be921685c2c538855c8dbc299177b008e8acaaf03aa10a954c29d2903c6f4




Output()

Accessing as ChorniBero15

Initialized MLflow to track repo "ChorniBero15/ML2"

Repository ChorniBero15/ML2 initialized!

# Cleaning

In [7]:
train_transaction_path = "/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv"
test_transaction_path = "/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv"
train_identity_path = "/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv"
test_identity_path = "/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv"

In [8]:
def reduce_mem_usage(df):
    start_mem = df.memory_usage(deep=True).sum() / 1024 ** 2
    
    for col in df.columns:
        col_type = df[col].dtype
        
        if pd.api.types.is_integer_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="integer")
        
        elif pd.api.types.is_float_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="float")
    
    end_mem = df.memory_usage(deep=True).sum() / 1024 ** 2
    reduction = 100 * (start_mem - end_mem) / start_mem
    
    print(f"Memory: {start_mem:.2f} MB -> {end_mem:.2f} MB")
    print(f"Reduced by {reduction:.2f}%")
    
    return df

In [9]:
with mlflow.start_run(run_name="DecisionTree_Cleaning"):
    train_transaction = reduce_mem_usage(pd.read_csv(train_transaction_path))
    test_transaction = reduce_mem_usage(pd.read_csv(test_transaction_path))
    
    train_identity = reduce_mem_usage(pd.read_csv(train_identity_path))
    test_identity = reduce_mem_usage(pd.read_csv(test_identity_path))
        
    test_transaction.columns = test_transaction.columns.str.replace("-", "_", regex=False)
    test_identity.columns = test_identity.columns.str.replace("-", "_", regex=False)
    
    train = train_transaction.merge(train_identity, on="TransactionID", how="left")
    test = test_transaction.merge(test_identity, on="TransactionID", how="left")
    
    train = reduce_mem_usage(train)
    test = reduce_mem_usage(test)
    
    mlflow.log_metric("train_rows", train.shape[0])
    mlflow.log_metric("train_columns", train.shape[1])
    mlflow.log_metric("test_rows", test.shape[0])
    mlflow.log_metric("test_columns", test.shape[1])
    mlflow.log_metric("fraud_rate", train["isFraud"].mean())
    mlflow.log_metric("train_missing_percent", train.isnull().mean().mean())
    mlflow.log_metric("test_missing_percent", test.isnull().mean().mean())
    
    del train_transaction, test_transaction, train_identity, test_identity
    gc.collect()

print("Train:", train.shape)
print("Test:", test.shape)

Memory: 2062.07 MB -> 1203.22 MB
Reduced by 41.65%
Memory: 1771.84 MB -> 1038.31 MB
Reduced by 41.40%
Memory: 143.14 MB -> 129.94 MB
Reduced by 9.22%
Memory: 140.08 MB -> 127.09 MB
Reduced by 9.27%
Memory: 1603.31 MB -> 1603.31 MB
Reduced by 0.00%
Memory: 1386.12 MB -> 1386.12 MB
Reduced by 0.00%
🏃 View run DecisionTree_Cleaning at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/3/runs/26797a99a1cd4f58a4e2d753a1151aa4
🧪 View experiment at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/3
Train: (590540, 434)
Test: (506691, 433)


# Feature Engineering

In [10]:
class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, cols=None):
        self.cols = cols if cols is not None else []
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        existing_cols = [col for col in self.cols if col in X.columns]
        return X.drop(columns=existing_cols)


class FeatureEngineering(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.created_cols = [
            "TransactionAmt_log",
            "TransactionAmt_decimal",
            "Transaction_day",
            "Transaction_hour",
            "Transaction_week",
            "missing_count",
            "has_identity",
            "email_domain_match",
            "card_missing_count"
        ]
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        X = X.drop(columns=[col for col in self.created_cols if col in X.columns], errors="ignore")
        
        features = pd.DataFrame(index=X.index)
        
        if "TransactionAmt" in X.columns:
            features["TransactionAmt_log"] = np.log1p(X["TransactionAmt"])
            features["TransactionAmt_decimal"] = ((X["TransactionAmt"] % 1) * 1000).round()
        
        if "TransactionDT" in X.columns:
            features["Transaction_day"] = X["TransactionDT"] // 86400
            features["Transaction_hour"] = (X["TransactionDT"] // 3600) % 24
            features["Transaction_week"] = features["Transaction_day"] // 7
        
        features["missing_count"] = X.isnull().sum(axis=1)
        
        if "id_01" in X.columns:
            features["has_identity"] = X["id_01"].notnull().astype("int8")
        else:
            features["has_identity"] = 0
        
        if "P_emaildomain" in X.columns and "R_emaildomain" in X.columns:
            features["email_domain_match"] = (X["P_emaildomain"] == X["R_emaildomain"]).astype("int8")
        else:
            features["email_domain_match"] = 0
        
        card_cols = [col for col in ["card1", "card2", "card3", "card4", "card5", "card6"] if col in X.columns]
        
        if len(card_cols) > 0:
            features["card_missing_count"] = X[card_cols].isnull().sum(axis=1)
        else:
            features["card_missing_count"] = 0
        
        return pd.concat([X, features], axis=1)


class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols=None):
        self.cols = cols
        self.freq_maps = {}
    
    def _normalize_series(self, s):
        return s.astype("object").where(s.notnull(), "__MISSING__")
    
    def fit(self, X, y=None):
        if self.cols is None:
            self.cols_ = X.select_dtypes(include=["object", "category"]).columns.tolist()
        else:
            self.cols_ = [col for col in self.cols if col in X.columns]
        
        self.freq_maps = {}
        
        for col in self.cols_:
            normalized = self._normalize_series(X[col])
            self.freq_maps[col] = normalized.value_counts(dropna=False).to_dict()
        
        return self
    
    def transform(self, X):
        X = X.copy()
        new_features = pd.DataFrame(index=X.index)
        
        for col in self.cols_:
            if col in X.columns:
                normalized = self._normalize_series(X[col])
                new_features[f"{col}_freq"] = normalized.map(self.freq_maps[col]).fillna(0).astype("float32")
        
        return pd.concat([X, new_features], axis=1)


class CategoricalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols=None):
        self.cols = cols
        self.category_maps = {}
    
    def _normalize_series(self, s):
        return s.astype("object").where(s.notnull(), "__MISSING__")
    
    def fit(self, X, y=None):
        if self.cols is None:
            self.cols_ = X.select_dtypes(include=["object", "category"]).columns.tolist()
        else:
            self.cols_ = [col for col in self.cols if col in X.columns]
        
        self.category_maps = {}
        
        for col in self.cols_:
            normalized = self._normalize_series(X[col])
            unique_values = pd.Series(normalized.unique())
            self.category_maps[col] = {value: idx for idx, value in enumerate(unique_values)}
        
        return self
    
    def transform(self, X):
        X = X.copy()
        
        for col in self.cols_:
            if col in X.columns:
                normalized = self._normalize_series(X[col])
                X[col] = normalized.map(self.category_maps[col]).fillna(-1).astype("int32")
        
        return X


class ReplaceInfValues(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        return X.replace([np.inf, -np.inf], np.nan)

# Feature Selection

In [11]:
class SimpleFeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, max_missing_ratio=0.95, min_unique_values=2):
        self.max_missing_ratio = max_missing_ratio
        self.min_unique_values = min_unique_values
    
    def fit(self, X, y=None):
        X = X.copy()
        
        numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
        X_numeric = X[numeric_cols]
        
        missing_ratio = X_numeric.isnull().mean()
        unique_counts = X_numeric.nunique(dropna=False)
        
        self.selected_features_ = [
            col for col in numeric_cols
            if missing_ratio[col] <= self.max_missing_ratio
            and unique_counts[col] >= self.min_unique_values
        ]
        
        self.dropped_features_ = [col for col in numeric_cols if col not in self.selected_features_]
        
        return self
    
    def transform(self, X):
        X = X.copy()
        
        for col in self.selected_features_:
            if col not in X.columns:
                X[col] = np.nan
        
        return X[self.selected_features_]


class TopCorrelationFeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, max_features=160, sample_size=120000, random_state=42):
        self.max_features = max_features
        self.sample_size = sample_size
        self.random_state = random_state
    
    def fit(self, X, y=None):
        X = X.copy()
        
        if y is None:
            self.selected_features_ = X.select_dtypes(include=[np.number]).columns.tolist()
            return self
        
        numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
        X_numeric = X[numeric_cols]
        y_series = pd.Series(y, index=X.index)
        
        if len(X_numeric) > self.sample_size:
            sample_index = X_numeric.sample(
                n=self.sample_size,
                random_state=self.random_state
            ).index
            
            X_numeric = X_numeric.loc[sample_index]
            y_series = y_series.loc[sample_index]
        
        medians = X_numeric.median(numeric_only=True).fillna(0)
        X_filled = X_numeric.fillna(medians)
        
        correlations = X_filled.corrwith(y_series).abs().fillna(0)
        correlations = correlations.sort_values(ascending=False)
        
        self.feature_scores_ = correlations.to_dict()
        self.selected_features_ = correlations.head(self.max_features).index.tolist()
        
        return self
    
    def transform(self, X):
        X = X.copy()
        
        for col in self.selected_features_:
            if col not in X.columns:
                X[col] = np.nan
        
        return X[self.selected_features_]


class MedianImputer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = X.copy()
        self.columns_ = X.columns.tolist()
        self.medians_ = X.median(numeric_only=True).fillna(0)
        return self
    
    def transform(self, X):
        X = X.copy()
        
        for col in self.columns_:
            if col not in X.columns:
                X[col] = np.nan
        
        X = X[self.columns_]
        X = X.fillna(self.medians_)
        
        return X

In [12]:
target = "isFraud"

X = train.drop(columns=[target])
y = train[target].astype("int8")

X_test = test.copy()

split_index = int(len(X) * 0.8)

X_train = X.iloc[:split_index].copy()
y_train = y.iloc[:split_index].copy()

X_val = X.iloc[split_index:].copy()
y_val = y.iloc[split_index:].copy()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("Train fraud rate:", y_train.mean())
print("Validation fraud rate:", y_val.mean())

X_train: (472432, 433)
X_val: (118108, 433)
Train fraud rate: 0.03513521522674162
Validation fraud rate: 0.034409184813899145


In [13]:
freq_encode_cols = [
    "card1", "card2", "card3", "card4", "card5", "card6",
    "addr1", "addr2",
    "P_emaildomain", "R_emaildomain",
    "DeviceType", "DeviceInfo",
    "ProductCD",
    "id_30", "id_31", "id_33"
]

In [14]:
with mlflow.start_run(run_name="DecisionTree_Feature_Engineering"):
    feature_engineering_pipeline = Pipeline([
        ("drop_columns", DropColumns(cols=["TransactionID"])),
        ("feature_engineering", FeatureEngineering()),
        ("frequency_encoding", FrequencyEncoder(cols=freq_encode_cols)),
        ("categorical_encoding", CategoricalEncoder()),
        ("replace_inf", ReplaceInfValues())
    ])
    
    feature_engineering_pipeline.fit(X_train, y_train)
    X_sample_fe = feature_engineering_pipeline.transform(X_train.iloc[:1000])
    
    mlflow.log_param("engineered_features", "amount,time,missingness,identity,email_match,frequency_encoding")
    mlflow.log_metric("features_after_engineering", X_sample_fe.shape[1])
    
    print("Features after engineering:", X_sample_fe.shape[1])

Features after engineering: 457
🏃 View run DecisionTree_Feature_Engineering at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/3/runs/ae235af9eef448bbb20692ec3026bc07
🧪 View experiment at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/3


In [15]:
with mlflow.start_run(run_name="DecisionTree_Feature_Selection"):
    feature_selection_pipeline = Pipeline([
        ("drop_columns", DropColumns(cols=["TransactionID"])),
        ("feature_engineering", FeatureEngineering()),
        ("frequency_encoding", FrequencyEncoder(cols=freq_encode_cols)),
        ("categorical_encoding", CategoricalEncoder()),
        ("replace_inf", ReplaceInfValues()),
        ("simple_feature_selection", SimpleFeatureSelector(max_missing_ratio=0.95, min_unique_values=2)),
        ("top_correlation_selection", TopCorrelationFeatureSelector(max_features=160, sample_size=120000)),
        ("median_imputer", MedianImputer())
    ])
    
    feature_selection_pipeline.fit(X_train, y_train)
    
    simple_selector = feature_selection_pipeline.named_steps["simple_feature_selection"]
    top_selector = feature_selection_pipeline.named_steps["top_correlation_selection"]
    selected_features = top_selector.selected_features_
    
    with open("decision_tree_selected_features.json", "w") as f:
        json.dump(selected_features, f, indent=2)
    
    with open("decision_tree_feature_scores.json", "w") as f:
        json.dump(top_selector.feature_scores_, f, indent=2)
    
    mlflow.log_param("feature_selection_method", "missing_constant_filter_plus_top_correlation")
    mlflow.log_param("max_missing_ratio", 0.95)
    mlflow.log_param("top_correlation_max_features", 160)
    mlflow.log_metric("features_after_simple_selection", len(simple_selector.selected_features_))
    mlflow.log_metric("features_after_top_correlation_selection", len(selected_features))
    mlflow.log_artifact("decision_tree_selected_features.json")
    mlflow.log_artifact("decision_tree_feature_scores.json")
    
    print("Features after simple selection:", len(simple_selector.selected_features_))
    print("Features after top correlation selection:", len(selected_features))

Features after simple selection: 450
Features after top correlation selection: 160
🏃 View run DecisionTree_Feature_Selection at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/3/runs/f97f90e7a8794974b7aa3ba3f288df35
🧪 View experiment at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/3


# Training

In [16]:
def build_decision_tree_pipeline(dt_params):
    return Pipeline([
        ("drop_columns", DropColumns(cols=["TransactionID"])),
        ("feature_engineering", FeatureEngineering()),
        ("frequency_encoding", FrequencyEncoder(cols=freq_encode_cols)),
        ("categorical_encoding", CategoricalEncoder()),
        ("replace_inf", ReplaceInfValues()),
        ("simple_feature_selection", SimpleFeatureSelector(max_missing_ratio=0.95, min_unique_values=2)),
        ("top_correlation_selection", TopCorrelationFeatureSelector(max_features=160, sample_size=120000)),
        ("median_imputer", MedianImputer()),
        ("model", DecisionTreeClassifier(**dt_params))
    ])

In [17]:
def run_decision_tree_experiment(run_name, dt_params):
    with mlflow.start_run(run_name=run_name):
        pipeline = build_decision_tree_pipeline(dt_params)
        pipeline.fit(X_train, y_train)
        
        train_pred_proba = pipeline.predict_proba(X_train)[:, 1]
        val_pred_proba = pipeline.predict_proba(X_val)[:, 1]
        
        train_roc_auc = roc_auc_score(y_train, train_pred_proba)
        val_roc_auc = roc_auc_score(y_val, val_pred_proba)
        train_pr_auc = average_precision_score(y_train, train_pred_proba)
        val_pr_auc = average_precision_score(y_val, val_pred_proba)
        overfit_gap = train_roc_auc - val_roc_auc
        
        val_pred = (val_pred_proba >= 0.5).astype(int)
        report = classification_report(y_val, val_pred)
        cm = confusion_matrix(y_val, val_pred)
        
        with open(f"{run_name}_classification_report.txt", "w") as f:
            f.write(report)
        
        pd.DataFrame(cm).to_csv(f"{run_name}_confusion_matrix.csv", index=False)
        
        mlflow.log_params(dt_params)
        mlflow.log_param("model_architecture", "DecisionTree")
        mlflow.log_param("validation_strategy", "time_based_80_20_split")
        mlflow.log_metric("train_roc_auc", train_roc_auc)
        mlflow.log_metric("validation_roc_auc", val_roc_auc)
        mlflow.log_metric("train_pr_auc", train_pr_auc)
        mlflow.log_metric("validation_pr_auc", val_pr_auc)
        mlflow.log_metric("overfit_gap", overfit_gap)
        mlflow.log_artifact(f"{run_name}_classification_report.txt")
        mlflow.log_artifact(f"{run_name}_confusion_matrix.csv")
        
        print(run_name)
        print("Train ROC-AUC:", train_roc_auc)
        print("Validation ROC-AUC:", val_roc_auc)
        print("Train PR-AUC:", train_pr_auc)
        print("Validation PR-AUC:", val_pr_auc)
        print("Overfit gap:", overfit_gap)
        
        return pipeline, {
            "run_name": run_name,
            "train_roc_auc": train_roc_auc,
            "validation_roc_auc": val_roc_auc,
            "train_pr_auc": train_pr_auc,
            "validation_pr_auc": val_pr_auc,
            "overfit_gap": overfit_gap
        }

In [18]:
underfit_params = {
    "criterion": "gini",
    "max_depth": 3,
    "min_samples_split": 200,
    "min_samples_leaf": 100,
    "max_features": "sqrt",
    "class_weight": "balanced",
    "random_state": 42
}

underfit_pipeline, underfit_metrics = run_decision_tree_experiment(
    "DecisionTree_Underfit_Model",
    underfit_params
)

DecisionTree_Underfit_Model
Train ROC-AUC: 0.7301854110657152
Validation ROC-AUC: 0.7352557760117866
Train PR-AUC: 0.15917220666984505
Validation PR-AUC: 0.17262711253389595
Overfit gap: -0.005070364946071382
🏃 View run DecisionTree_Underfit_Model at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/3/runs/f5da65f199c5408c9c85216bcdf1506b
🧪 View experiment at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/3


In [19]:
overfit_params = {
    "criterion": "gini",
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": None,
    "class_weight": "balanced",
    "random_state": 42
}

overfit_pipeline, overfit_metrics = run_decision_tree_experiment(
    "DecisionTree_Overfit_Model",
    overfit_params
)

DecisionTree_Overfit_Model
Train ROC-AUC: 0.994975277693746
Validation ROC-AUC: 0.6048503733587974
Train PR-AUC: 0.9404963931003442
Validation PR-AUC: 0.12462518309554547
Overfit gap: 0.3901249043349486
🏃 View run DecisionTree_Overfit_Model at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/3/runs/b6d8cecc34fa480ba5547c53890413cb
🧪 View experiment at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/3


In [20]:
regularized_params = {
    "criterion": "entropy",
    "max_depth": 8,
    "min_samples_split": 80,
    "min_samples_leaf": 30,
    "max_features": "sqrt",
    "class_weight": "balanced",
    "random_state": 42
}

regularized_pipeline, regularized_metrics = run_decision_tree_experiment(
    "DecisionTree_Regularized_Model",
    regularized_params
)

DecisionTree_Regularized_Model
Train ROC-AUC: 0.7931526370463992
Validation ROC-AUC: 0.7755934456210022
Train PR-AUC: 0.33075473333722644
Validation PR-AUC: 0.302164213616349
Overfit gap: 0.01755919142539697
🏃 View run DecisionTree_Regularized_Model at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/3/runs/dfd9914a8c834d38852d17b5e43c3b77
🧪 View experiment at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/3


In [23]:
dt_base_params = {
    "class_weight": "balanced",
    "random_state": 42
}

dt_param_grid = {
    "model__criterion": ["gini", "entropy"],
    "model__max_depth": [5, 8],
    "model__min_samples_split": [50, 100],
    "model__min_samples_leaf": [20, 50],
    "model__max_features": ["sqrt"]
}

candidate_count = 1
for values in dt_param_grid.values():
    candidate_count *= len(values)

print("Grid candidate count:", candidate_count)

grid_search_pipeline = build_decision_tree_pipeline(dt_base_params)

time_cv = TimeSeriesSplit(n_splits=2)

grid_search = GridSearchCV(
    estimator=grid_search_pipeline,
    param_grid=dt_param_grid,
    scoring="roc_auc",
    cv=time_cv,
    n_jobs=1,
    verbose=2,
    return_train_score=True,
    refit=True
)

with mlflow.start_run(run_name="DecisionTree_Grid_Search"):
    grid_search.fit(X_train, y_train)
    
    grid_results_df = pd.DataFrame(grid_search.cv_results_)
    grid_results_df.to_csv("decision_tree_grid_search_results.csv", index=False)
    
    train_pred_proba = grid_search.best_estimator_.predict_proba(X_train)[:, 1]
    val_pred_proba = grid_search.best_estimator_.predict_proba(X_val)[:, 1]
    
    grid_train_roc_auc = roc_auc_score(y_train, train_pred_proba)
    grid_val_roc_auc = roc_auc_score(y_val, val_pred_proba)
    grid_train_pr_auc = average_precision_score(y_train, train_pred_proba)
    grid_val_pr_auc = average_precision_score(y_val, val_pred_proba)
    grid_overfit_gap = grid_train_roc_auc - grid_val_roc_auc
    
    with open("decision_tree_param_grid.json", "w") as f:
        json.dump(dt_param_grid, f, indent=2)
    
    mlflow.log_param("search_method", "GridSearchCV")
    mlflow.log_param("cv_strategy", "TimeSeriesSplit")
    mlflow.log_param("cv_splits", time_cv.n_splits)
    mlflow.log_param("scoring", "roc_auc")
    mlflow.log_param("candidate_count", candidate_count)
    
    for param_name, param_value in grid_search.best_params_.items():
        mlflow.log_param(f"best_{param_name}", param_value)
    
    mlflow.log_metric("best_cv_roc_auc", grid_search.best_score_)
    mlflow.log_metric("grid_train_roc_auc", grid_train_roc_auc)
    mlflow.log_metric("grid_validation_roc_auc", grid_val_roc_auc)
    mlflow.log_metric("grid_train_pr_auc", grid_train_pr_auc)
    mlflow.log_metric("grid_validation_pr_auc", grid_val_pr_auc)
    mlflow.log_metric("grid_overfit_gap", grid_overfit_gap)
    
    mlflow.log_artifact("decision_tree_param_grid.json")
    mlflow.log_artifact("decision_tree_grid_search_results.csv")

best_params_from_grid = dt_base_params.copy()

for param_name, param_value in grid_search.best_params_.items():
    clean_name = param_name.replace("model__", "")
    best_params_from_grid[clean_name] = param_value

grid_pipeline = grid_search.best_estimator_
best_params_for_grid = best_params_from_grid.copy()

grid_metrics = {
    "run_name": "DecisionTree_Grid_Search",
    "best_cv_roc_auc": grid_search.best_score_,
    "train_roc_auc": grid_train_roc_auc,
    "validation_roc_auc": grid_val_roc_auc,
    "train_pr_auc": grid_train_pr_auc,
    "validation_pr_auc": grid_val_pr_auc,
    "overfit_gap": grid_overfit_gap
}

print("Best grid search params:", grid_search.best_params_)
print("Best CV ROC-AUC:", grid_search.best_score_)
print("Grid validation ROC-AUC:", grid_val_roc_auc)
print("Grid validation PR-AUC:", grid_val_pr_auc)
print("Grid overfit gap:", grid_overfit_gap)

Grid candidate count: 16
Fitting 2 folds for each of 16 candidates, totalling 32 fits
[CV] END model__criterion=gini, model__max_depth=5, model__max_features=sqrt, model__min_samples_leaf=20, model__min_samples_split=50; total time=  14.9s
[CV] END model__criterion=gini, model__max_depth=5, model__max_features=sqrt, model__min_samples_leaf=20, model__min_samples_split=50; total time=  23.1s
[CV] END model__criterion=gini, model__max_depth=5, model__max_features=sqrt, model__min_samples_leaf=20, model__min_samples_split=100; total time=  14.9s
[CV] END model__criterion=gini, model__max_depth=5, model__max_features=sqrt, model__min_samples_leaf=20, model__min_samples_split=100; total time=  23.0s
[CV] END model__criterion=gini, model__max_depth=5, model__max_features=sqrt, model__min_samples_leaf=50, model__min_samples_split=50; total time=  14.9s
[CV] END model__criterion=gini, model__max_depth=5, model__max_features=sqrt, model__min_samples_leaf=50, model__min_samples_split=50; total t

In [25]:
results_df = pd.DataFrame([
    underfit_metrics,
    overfit_metrics,
    regularized_metrics,
    grid_metrics
])

results_df = results_df.sort_values("validation_roc_auc", ascending=False)
results_df.to_csv("decision_tree_experiment_results.csv", index=False)

results_df

,run_name,train_roc_auc,validation_roc_auc,train_pr_auc,validation_pr_auc,overfit_gap,best_cv_roc_auc
2,DecisionTree_Regularized_Model,0.793153,0.775593,0.330755,0.302164,0.017559,NaN
3,DecisionTree_Grid_Search,0.792077,0.772060,0.330387,0.304210,0.020017,0.796608
0,DecisionTree_Underfit_Model,0.730185,0.735256,0.159172,0.172627,-0.005070,NaN
1,DecisionTree_Overfit_Model,0.994975,0.604850,0.940496,0.124625,0.390125,NaN


In [26]:
with mlflow.start_run(run_name="DecisionTree_Model_Comparison"):
    mlflow.log_artifact("decision_tree_experiment_results.csv")
    
    mlflow.log_param("best_run_name", results_df.iloc[0]["run_name"])
    mlflow.log_metric("best_validation_roc_auc", results_df.iloc[0]["validation_roc_auc"])
    mlflow.log_metric("best_validation_pr_auc", results_df.iloc[0]["validation_pr_auc"])
    mlflow.log_metric("best_overfit_gap", results_df.iloc[0]["overfit_gap"])

print("Best run:", results_df.iloc[0]["run_name"])

🏃 View run DecisionTree_Model_Comparison at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/3/runs/c64eb34441224910886a4667b808c1b5
🧪 View experiment at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/3
Best run: DecisionTree_Regularized_Model


In [27]:
best_run_name = results_df.iloc[0]["run_name"]

if best_run_name == "DecisionTree_Underfit_Model":
    best_params_for_final = underfit_params.copy()
elif best_run_name == "DecisionTree_Overfit_Model":
    best_params_for_final = overfit_params.copy()
elif best_run_name == "DecisionTree_Regularized_Model":
    best_params_for_final = regularized_params.copy()
else:
    best_params_for_final = best_params_for_grid.copy()

print("Best run:", best_run_name)
print(best_params_for_final)

Best run: DecisionTree_Regularized_Model
{'criterion': 'entropy', 'max_depth': 8, 'min_samples_split': 80, 'min_samples_leaf': 30, 'max_features': 'sqrt', 'class_weight': 'balanced', 'random_state': 42}


In [28]:
final_pipeline = build_decision_tree_pipeline(best_params_for_final)

with mlflow.start_run(run_name="DecisionTree_Final_Pipeline"):
    final_pipeline.fit(X, y)
    
    full_train_pred = final_pipeline.predict_proba(X)[:, 1]
    full_train_roc_auc = roc_auc_score(y, full_train_pred)
    full_train_pr_auc = average_precision_score(y, full_train_pred)
    
    mlflow.log_params(best_params_for_final)
    mlflow.log_param("selected_best_run", best_run_name)
    mlflow.log_param("model_saved_as", "sklearn_pipeline")
    mlflow.log_metric("full_train_roc_auc", full_train_roc_auc)
    mlflow.log_metric("full_train_pr_auc", full_train_pr_auc)
    
    pipeline = final_pipeline
    
    with open("model.pkl", "wb") as f:
        pickle.dump(pipeline, f)
    
    mlflow.log_artifact("model.pkl")
    mlflow.log_artifact("decision_tree_experiment_results.csv")
    
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="model",
        registered_model_name=REGISTERED_MODEL_NAME
    )
    
    print("Final Decision Tree pipeline saved and registered.")
    print("Full train ROC-AUC:", full_train_roc_auc)
    print("Full train PR-AUC:", full_train_pr_auc)

2026/05/06 16:33:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 16:33:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'model_decision_tree'.
2026/05/06 16:33:39 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: model_decision_tree, version 1
Created version '1' of model 'model_decision_tree'.


Final Decision Tree pipeline saved and registered.
Full train ROC-AUC: 0.8036013205766313
Full train PR-AUC: 0.3357119109183075
🏃 View run DecisionTree_Final_Pipeline at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/3/runs/a37638e681354d8b86eca834fa7b5ec8
🧪 View experiment at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/3


In [29]:
test_ids = X_test["TransactionID"].copy()

y_proba = final_pipeline.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    "TransactionID": test_ids,
    "isFraud": y_proba
})

submission.to_csv("submission_decision_tree.csv", index=False)

print(submission.shape)
submission.head()

(506691, 2)


,TransactionID,isFraud
0,3663549,0.435151
1,3663550,0.386859
2,3663551,0.386859
3,3663552,0.165778
4,3663553,0.107903


In [30]:
with mlflow.start_run(run_name="DecisionTree_Submission"):
    mlflow.log_artifact("submission_decision_tree.csv")
    mlflow.log_metric("submission_rows", submission.shape[0])

🏃 View run DecisionTree_Submission at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/3/runs/cd11a0b3b3204ea4ae970de67f57635b
🧪 View experiment at: https://dagshub.com/ChorniBero15/ML2.mlflow/#/experiments/3
